#6 - Gold Histórico - Histórico e indicadores

---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida
definimos os caminhos das tabelas que serão utilizadas durante a execução.

#.
### Variáveis e suas utilizações
silver_historico = define o caminho de leitura da tabela silver do histórico

gold_historico = define o caminho onde será criada a tabela gold do histórico diário

gold_indicadores = define o caminho onde será criada a tabela de indicadores históricos

---

In [0]:
import pyspark.sql.functions as sf

silver_historico = "workspace.stocks.silver"

gold_historico = "workspace.stocks.gold"
gold_indicadores = "workspace.stocks.gold_indicadores"


---

Lemos a tabela silver do histórico e padronizamos os valores numéricos aplicando round de duas casas decimais em todas as colunas de preço e variação.

Convertemos event_time para data, removendo a informação de horário que não é relevante para análises de granularidade diária.

Salvamos o resultado na tabela gold do histórico no formato Delta.

---

In [0]:
df_hist = spark.read.table(silver_historico)

df_gold_hist = (df_hist.withColumn("event_time", sf.to_date("event_time"))
      .withColumn("open", sf.round(sf.col("open"),2))
      .withColumn("high", sf.round(sf.col("high"),2))
      .withColumn("low", sf.round(sf.col("low"),2))
      .withColumn("close", sf.round(sf.col("close"),2))
      .withColumn("variacao_real", sf.round(sf.col("variacao_real"),2))
      .withColumn("variacao_percent", sf.round(sf.col("variacao_percent"),2)))

(
  df_gold_hist.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_historico)
)
print("-Dados gravados-")

df_gold_hist.show(10)

---

Geramos uma tabela de indicadores agregados por ticker com base em todo o histórico disponível.

Os indicadores cobrem o período total calculado, média, menor e maior fechamento, média, menor e maior variação percentual e volume total negociado.

Essa tabela serve como baseline para comparação com os dados em tempo real do micro batch.

---

In [0]:
df_indicadores = (df_hist.groupBy("ticker")
      .agg(
            sf.count("ticker").alias("dias_calculados"),
            sf.to_date(sf.min("event_time")).alias("primeiro_dia"),
            sf.to_date(sf.max("event_time")).alias("ultimo_dia"),
            sf.round(sf.avg("close"),2).alias("media_fechamento"),
            sf.round(sf.min("close"),2).alias("menor_fechamento"),
            sf.round(sf.max("close"),2).alias("maior_fechamento"),
            sf.round(sf.avg("variacao_percent"),2).alias("media_porcentagem_variacao"),
            sf.round(sf.min("variacao_percent"),2).alias("menor_porcentagem_variacao"),
            sf.round(sf.max("variacao_percent"),2).alias("maior_porcentagem_variacao"),
            sf.sum("volume").alias("volume_total")
      ))

(
    df_indicadores.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_indicadores)
)
print("-Dados gravados-")

display(df_indicadores)